# 04 · Training on Google Colab

This notebook reproduces all modelling experiments of the project on a Colab GPU by running `src/train.py`.

**Setup**
1. Select a GPU runtime: `Runtime → Change runtime type → GPU`.
2. Run the cells in order.

Results are written to Google Drive (`OUTPUT_DIR`). Every finished run is saved immediately, so if the runtime
disconnects, running the notebook again continues from the last finished run.

| Stage | Training data | Evaluation data | Runs |
|---|---|---|---|
| `test` | Jan–Aug (early stopping on Sep–Oct) | Nov–Dec, used once | 3 horizons × (2 baselines + XGBoost and LSTM × 5 feature sets × 3 seeds) |
| `walkforward` | months 1 … m | month m + 1 (May–Oct) | 3 horizons × 6 folds × (2 baselines + XGBoost and LSTM × 2 feature sets) |

> **Run record.** This copy contains the outputs of the run used in the project (NVIDIA L4 GPU, October 2026). Library warnings were removed from the logs; every run line and result is unchanged. The test stage was resumed once after a runtime restart, so its log starts at run 47 of 96 (all 96 results are in `results/results.csv`).

## 1 · Settings

In [1]:
SOURCE='zip'          # 'github' to clone the repository, 'zip' to use a copy of the project stored in Google Drive
REPO_URL='https://github.com/sfeyzayildiz/mahmutbey-traffic-forecast.git'
ZIP_PATH='/content/drive/MyDrive/mahmutbey-traffic-forecast.zip'
OUTPUT_DIR='/content/drive/MyDrive/mahmutbey_results'

## 2 · Connect Google Drive and check the GPU

In [2]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

Mounted at /content/drive
name, memory.total [MiB]
NVIDIA L4, 23034 MiB


## 3 · Get the project

In [3]:
import os
PROJECT='/content/mahmutbey-traffic-forecast'
!rm -rf $PROJECT
if SOURCE=='github':
    !git clone -q $REPO_URL $PROJECT
else:
    !unzip -q $ZIP_PATH -d /content/
os.chdir(PROJECT)
os.makedirs(OUTPUT_DIR, exist_ok=True)
!ls

data  figures  LICENSE	notebooks  requirements.txt  src


## 4 · Install dependencies

In [4]:
!pip install -q -r requirements.txt
import tensorflow as tf, xgboost
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))
print('XGBoost', xgboost.__version__)

XGBoost 3.4.1


## 5 · Build the processed dataset
Runs the processing pipeline of notebook 02 and writes `data/processed/dataset.parquet`.

In [5]:
import sys; sys.path.insert(0, '.')
from src.data import load_traffic, load_weather, load_fuel_events, ROOT
from src.processing import build_dataset
df, meta, _=build_dataset(load_traffic(), load_weather(), load_fuel_events())
(ROOT/'data'/'processed').mkdir(parents=True, exist_ok=True)
df.to_parquet(ROOT/'data'/'processed'/'dataset.parquet')
print(df.shape, meta)

(8784, 33) {'suspect_ratio': 0.1, 'neighbours': ['sxk3xd', 'sxk985', 'sxk3xg', 'sxk3x7', 'sxk3x9', 'sxk3x8']}


## 6 · Smoke test
A short run (one seed, a few LSTM epochs) to check the environment. Results go to `results_quick.csv`
and are not used in the analysis.

In [6]:
!python -m src.train --out $OUTPUT_DIR --quick

30 runs planned, 30 already done
Done. Results: /content/drive/MyDrive/mahmutbey_results/results_quick.csv


## 7 · Test stage
Each line of the log shows the run, its test MAE, the coverage of the 90% interval and the training time.

In [7]:
!python -m src.train --out $OUTPUT_DIR --stage test

96 runs planned, 46 already done
[47/96] test|None|xgboost|h2|no_fuel|s1               MAE 3.10  PICP90 85.0  (8.4s)
[48/96] test|None|xgboost|h2|no_fuel|s2               MAE 3.11  PICP90 85.3  (7.1s)
[49/96] test|None|xgboost|h2|no_fuel|s3               MAE 3.11  PICP90 84.7  (7.6s)
[50/96] test|None|lstm|h2|full|s1                     MAE 3.13  PICP90 77.1  (66.9s)
[51/96] test|None|lstm|h2|full|s2                     MAE 3.23  PICP90 84.8  (60.4s)
[52/96] test|None|lstm|h2|full|s3                     MAE 3.15  PICP90 81.7  (75.8s)
[53/96] test|None|lstm|h2|no_neighbours|s1            MAE 3.26  PICP90 83.9  (48.8s)
[54/96] test|None|lstm|h2|no_neighbours|s2            MAE 3.37  PICP90 84.9  (61.6s)
[55/96] test|None|lstm|h2|no_neighbours|s3            MAE 3.35  PICP90 79.8  (49.4s)
[56/96] test|None|lstm|h2|no_calendar|s1              MAE 3.33  PICP90 78.5  (49.7s)
[57/96] test|None|lstm|h2|no_calendar|s2              MAE 3.31  PICP90 79.2  (58.8s)
[58/96] test|None|lstm|h2|no_calend

## 8 · Walk-forward stage

In [8]:
!python -m src.train --out $OUTPUT_DIR --stage walkforward

108 runs planned, 96 already done
[1/108] walkforward|5|persistence|h1|full|s0          MAE 5.06  PICP90 90.4  (0.0s)
[2/108] walkforward|5|seasonal_naive|h1|full|s0       MAE 5.10  PICP90 91.8  (0.0s)
[3/108] walkforward|5|xgboost|h1|full|s1              MAE 2.87  PICP90 79.6  (6.9s)
[4/108] walkforward|5|xgboost|h1|no_neighbours|s1     MAE 2.95  PICP90 75.8  (6.4s)
[5/108] walkforward|5|lstm|h1|full|s1                 MAE 3.06  PICP90 69.6  (25.5s)
[6/108] walkforward|5|lstm|h1|no_neighbours|s1        MAE 3.02  PICP90 67.3  (21.4s)
[7/108] walkforward|5|persistence|h2|full|s0          MAE 8.51  PICP90 90.6  (0.0s)
[8/108] walkforward|5|seasonal_naive|h2|full|s0       MAE 5.10  PICP90 91.8  (0.0s)
[9/108] walkforward|5|xgboost|h2|full|s1              MAE 3.54  PICP90 73.4  (6.8s)
[10/108] walkforward|5|xgboost|h2|no_neighbours|s1     MAE 3.63  PICP90 75.8  (6.4s)
[11/108] walkforward|5|lstm|h2|full|s1                 MAE 3.93  PICP90 66.0  (17.7s)
[12/108] walkforward|5|lstm|h2|no_nei

## 9 · Summary and export
Shows the mean test results of the full models and writes `mahmutbey_results.zip`
(results table, predictions and MLflow database) next to `OUTPUT_DIR`.

In [9]:
import shutil
from src.train import load_results
r=load_results(f'{OUTPUT_DIR}/results.csv')
print(f'{len(r)} runs finished')
display(r.query("stage=='test' and ablation=='full'")
         .groupby(['horizon', 'model'])[['MAE', 'RMSE', 'MAPE', 'R2', 'PICP90', 'MPIW90']].mean().round(2))
shutil.make_archive(OUTPUT_DIR, 'zip', OUTPUT_DIR)
print(f'Saved {OUTPUT_DIR}.zip')

MAE   RMSE   MAPE    R2  PICP90  MPIW90
horizon model                                                    
1       lstm             2.71   3.67   6.51  0.96   80.92    9.53
        persistence      5.14   7.23  12.20  0.83   89.99   24.00
        seasonal_naive   4.62   6.65  11.21  0.85   96.24   33.00
        xgboost          2.72   3.72   6.65  0.95   81.56    9.39
2       lstm             3.17   4.37   7.60  0.94   81.21   10.89
        persistence      8.41  11.71  19.89  0.55   89.11   40.00
        seasonal_naive   4.62   6.65  11.21  0.85   96.24   33.00
        xgboost          3.10   4.27   7.60  0.94   85.48   11.89
3       lstm             3.42   4.73   8.24  0.93   76.62   10.50
        persistence     11.49  15.38  27.20  0.22   91.03   54.00
        seasonal_naive   4.62   6.65  11.21  0.85   96.24   33.00
        xgboost          3.32   4.60   8.08  0.93   84.71   12.69

Saved /content/drive/MyDrive/mahmutbey_results.zip


## 10 · Predictions for interval calibration
Retrains the full XGBoost and LSTM models (same setup and seeds as the test stage) and saves their quantile
predictions for the validation and test periods. The validation predictions are used to calibrate the
prediction intervals in notebook 05. Output: `results/calibration/`.

In [11]:
!python -m src.export_predictions --out $OUTPUT_DIR
import shutil
shutil.make_archive('/content/drive/MyDrive/mahmutbey_calibration', 'zip', f'{OUTPUT_DIR}/calibration')
print('Saved MyDrive/mahmutbey_calibration.zip')

saved xgboost_h1_s1.parquet
saved xgboost_h1_s2.parquet
saved xgboost_h1_s3.parquet
saved lstm_h1_s1.parquet
saved lstm_h1_s2.parquet
saved lstm_h1_s3.parquet
saved xgboost_h2_s1.parquet
saved xgboost_h2_s2.parquet
saved xgboost_h2_s3.parquet
saved lstm_h2_s1.parquet
saved lstm_h2_s2.parquet
saved lstm_h2_s3.parquet
saved xgboost_h3_s1.parquet
saved xgboost_h3_s2.parquet
saved xgboost_h3_s3.parquet
saved lstm_h3_s1.parquet
saved lstm_h3_s2.parquet
saved lstm_h3_s3.parquet
Saved MyDrive/mahmutbey_calibration.zip
